# Tiền xử lý dữ liệu Titanic (Preprocessing)

## Improt thư viện

In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## 1. Mục tiêu tiền xử lý
+ **Mô tả**:
    + Xử lý missing values (Age, Embarked, Cabin).
    + Encoding categorical features (Sex, Embarked).
    + Feature engineering (tạo FamilySize từ SibSp + Parch).
    + Scaling numerical features (Age, Fare) nếu cần.
+ **Dữ liệu vào**: Từ interim sau EDA.
+ **Kết quả**: Dữ liệu sạch, lưu vào processed.

## 2. Load dữ liệu từ interim

In [4]:
# Load từ interim
train_path = '../data/interim/train_eda.csv'
test_path = '../data/interim/test_eda.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Kết hợp để xử lý chung (test không có Survived)
df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)

## 3. Xử lý missing values
+ Fill Age bằng mean.
+ Fill Embarked bằng mode.
+ Drop Cabin (quá nhiều missing).
+ Fill Fare (nếu có) bằng mean.

In [5]:
# Kiểm tra missing
print(df_all.isnull().sum())

# Fill Age và Fare bằng mean
imputer_num = SimpleImputer(strategy='mean')
df_all[['Age', 'Fare']] = imputer_num.fit_transform(df_all[['Age', 'Fare']])

# Fill Embarked bằng mode
imputer_cat = SimpleImputer(strategy='most_frequent')
df_all['Embarked'] = imputer_cat.fit_transform(df_all[['Embarked']]).ravel()

# Drop Cabin, Ticket, Name (tạm drop, nếu engineering thì giữ Name)
df_all.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)

# Kiểm tra lại missing
print(df_all.isnull().sum())

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
dtype: int64
Survived    418
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64


## 4. Encoding categorical features
+ Sex: male=0, female=1.
+ Embarked: one-hot encoding.

In [6]:
# Encoding Sex
df_all['Sex'] = df_all['Sex'].map({'male': 0, 'female': 1})

# One-hot Embarked
df_all = pd.get_dummies(df_all, columns=['Embarked'], drop_first=True)

## 5. Feature engineering
+ Tạo FamilySize = SibSp + Parch + 1.
+ (Optional: Trích Title từ Name nếu giữ Name).

In [7]:
# Tạo FamilySize
df_all['FamilySize'] = df_all['SibSp'] + df_all['Parch'] + 1

# Drop SibSp và Parch nếu không cần
# df_all.drop(['SibSp', 'Parch'], axis=1, inplace=True)

## 6. Scaling numerical features
+ Scale Age và Fare để model tốt hơn.

In [8]:
scaler = StandardScaler()
df_all[['Age', 'Fare']] = scaler.fit_transform(df_all[['Age', 'Fare']])

## 7. Lưu dữ liệu đã xử lý
+ Tách lại train/test và lưu vào processed.

In [9]:
# Tách train và test (train có Survived not null)
train_processed = df_all[df_all['Survived'].notnull()]
test_processed = df_all[df_all['Survived'].isnull()].drop('Survived', axis=1)

# Lưu
train_processed.to_csv('../data/processed/train_processed.csv', index=False)
test_processed.to_csv('../data/processed/test_processed.csv', index=False)
print("Data saved to processed/")

Data saved to processed/


# Kết thúc

In [10]:
# Cách an toàn, đa nền tảng để xóa HTML hiện có và export notebook sang HTML
import os
import subprocess
from pathlib import Path
# Tính toán đường dẫn notebook và output tương đối với file notebook này
nb_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')
# Nếu chạy bên trong notebook, sử dụng thư mục làm việc hiện tại của server notebook
nb_dir = nb_dir if nb_dir.exists() else Path('.')
nb_path = nb_dir / 'preprocessing.ipynb'
out_path = nb_dir / 'preprocessing.html'
# Xóa file output hiện có nếu tồn tại
if out_path.exists():
    print(f'Removing existing file: {out_path}')
    out_path.unlink()
# Chạy nbconvert sử dụng subprocess để ổn định trên Windows
cmd = ['jupyter', 'nbconvert', str(nb_path), '--to', 'html']
print('Running:', ' '.join(cmd))
try:
    subprocess.run(cmd, check=True)
    print('Export complete:', out_path)
except subprocess.CalledProcessError as e:
    print('nbconvert failed with returncode', e.returncode)
    print('Ensure jupyter is available in the PATH of the environment running this notebook.')

Running: jupyter nbconvert preprocessing.ipynb --to html
Export complete: preprocessing.html
